# JAXOOM long T4 allocator-residency campaign

This notebook searches workload scale for missing families first, then performs allocator-capacity thresholding. It is bounded and checkpointed.

In [ ]:
import json, sys, importlib, zipfile, collections, statistics
from pathlib import Path
REPO=Path("/content/jaxoom")
PINNED_COMMIT="7dbf22056cef25b0d77ecf90b453cfe5cac4cebb"


## Install and pin infrastructure

In [ ]:
!pip install -q "jax[cuda12]==0.11.0"
!rm -rf /content/jaxoom
!git clone -q https://github.com/Slavov88/jaxoom.git /content/jaxoom
!cd /content/jaxoom && git checkout -q 7dbf22056cef25b0d77ecf90b453cfe5cac4cebb && pip install -q -e .


## Verify T4 and JAX

In [ ]:
sys.path.insert(0,"/content/jaxoom/src")
sys.modules.pop("jaxoom",None); importlib.invalidate_caches()
import jax,jaxlib,jaxoom
assert jax.__version__=="0.11.0" and jaxlib.__version__=="0.11.0"
assert jax.default_backend()=="gpu"
device=jax.devices()[0]
assert "T4" in getattr(device,"device_kind",str(device)),device
environment={"device":str(device),"device_kind":getattr(device,"device_kind",None),"jax_version":jax.__version__,"jaxlib_version":jaxlib.__version__,"backend":jax.default_backend(),"pinned_commit":PINNED_COMMIT,"preallocate":False}
Path("/content/environment.json").write_text(json.dumps(environment,indent=2)+"
")
print(json.dumps(environment,indent=2))


## Calibrate low allocator limits

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --calibrate --fractions 0.04,0.05,0.06,0.07,0.08,0.09,0.10,0.12,0.15,0.20,0.25,0.30,0.35 --output /content/fraction_map.json --timeout 30


## Stage A: family-directed workload-scale search

This uses one low allocator capacity and searches scale. It does not treat scale-search OOM as an allocator threshold label.

In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_long_campaign.py --scale-output /content/scale_searches.json --selection-output /content/candidate_configurations.json --fraction 0.06 --timeout 60 --max-probes 100 --max-per-family 3


## Stage B: allocator-capacity thresholding of selected configurations

In [ ]:
selection=json.loads(Path("/content/candidate_configurations.json").read_text())
manifest={"status":"MANIFEST","candidates":selection["selected"]}
Path("/content/threshold_manifest.json").write_text(json.dumps(manifest,indent=2)+"
")
print(selection.get("selected_family_counts"))


In [ ]:
!cd /content/jaxoom && PYTHONPATH=src:experiments python experiments/t4_allocator_residency.py --manifest /content/threshold_manifest.json --fraction-map /content/fraction_map.json --output /content/raw_probes.json --timeout 60 --max-probes 80


## Derive thresholds and freeze the dataset

In [ ]:
def outcome(row): return (row.get("execution_statuses") or [row.get("outcome") or row.get("status") or row.get("compile_status") or "OTHER_FAILURE"])[0]
def cap(row): return ((row.get("snapshots") or [{}])[0]).get("allocator_limit_bytes")
def derive(rows):
    groups={}
    for row in rows: groups.setdefault(row.get("configuration_id"),[]).append(row)
    out=[]
    for cid,g in groups.items():
        fits=[r for r in g if outcome(r)=="FIT" and cap(r) is not None]
        ooms=[r for r in g if outcome(r) in {"COMPILE_OOM","EXECUTION_OOM"} and cap(r) is not None]
        if fits and ooms:
            lo=max(cap(r) for r in ooms); hi=min(cap(r) for r in fits)
            out.append({"configuration_id":cid,"family":g[0].get("family"),"configuration":g[0].get("configuration"),"dtype":g[0].get("dtype"),"known_oom_capacity":lo,"known_fit_capacity":hi,"bracket_width_bytes":hi-lo})
    return out
raw=json.loads(Path("/content/raw_probes.json").read_text()).get("rows",[])
new=derive(raw)
old=json.loads((REPO/"experiments/allocator_residency_t4_refined_thresholds_2026-09-09.json").read_text())["rows"]
all_thresholds=old+new
Path("/content/new_thresholds.json").write_text(json.dumps({"status":"OBSERVED","rows":new},indent=2)+"
")
Path("/content/all_t4_thresholds.json").write_text(json.dumps({"status":"OBSERVED","rows":all_thresholds},indent=2)+"
")
families=collections.Counter(r["family"] for r in all_thresholds); dtypes=collections.Counter(r["dtype"] for r in all_thresholds)
under=sum(families.get(f,0) for f in ("mlp","training","autodiff"))
gate=len(all_thresholds)>=15 and len(families)>=4 and under>=2
summary={"status":"OBSERVED","new_thresholds":len(new),"total_thresholds":len(all_thresholds),"families":dict(families),"dtypes":dict(dtypes),"outcomes":dict(collections.Counter(outcome(r) for r in raw)),"model_gate":{"met":gate,"threshold_count":len(all_thresholds),"family_count":len(families),"underrepresented_count":under}}
Path("/content/threshold_summary.json").write_text(json.dumps(summary,indent=2)+"
")
not_run={"status":"NOT_RUN","reason":"modeling_gate_not_met" if not gate else "REQUIRES_OFFLINE_MODELING_PASS"}
Path("/content/model_comparison.json").write_text(json.dumps(not_run,indent=2)+"
")
Path("/content/family_holdout.json").write_text(json.dumps({"status":"NOT_RUN","reason":"modeling_gate_not_met" if not gate else "REQUIRES_OFFLINE_MODELING_PASS"},indent=2)+"
")
# Paired comparison uses only IDs present in committed RTX evidence.
rtx={r["configuration_id"]:r for r in json.loads((REPO/"experiments/allocator_residency_thresholds_2026-09-09.json").read_text())["useful_thresholds"]}
paired=[{"configuration_id":r["configuration_id"],"family":r["family"],"dtype":r["dtype"],"t4_upper":r["known_fit_capacity"],"rtx_upper":rtx[r["configuration_id"]]["required_allocator_upper_bytes"],"upper_ratio":r["known_fit_capacity"]/rtx[r["configuration_id"]]["required_allocator_upper_bytes"]} for r in all_thresholds if r["configuration_id"] in rtx]
Path("/content/paired_rtx_t4.json").write_text(json.dumps({"status":"OBSERVED","rows":paired},indent=2)+"
")
print(json.dumps(summary,indent=2))


## Package results

In [ ]:
files=["environment.json","fraction_map.json","scale_searches.json","candidate_configurations.json","raw_probes.json","new_thresholds.json","all_t4_thresholds.json","threshold_summary.json","model_comparison.json","family_holdout.json","paired_rtx_t4.json"]
with zipfile.ZipFile("/content/jaxoom_t4_allocator_residency_long_campaign.zip","w",compression=zipfile.ZIP_DEFLATED) as z:
    for name in files: z.write("/content/"+name,arcname=name)
from google.colab import files
files.download("/content/jaxoom_t4_allocator_residency_long_campaign.zip")
